In [1]:
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# -----------------------------
# Helper functions
# -----------------------------
def parse_oha_file(filepath):
    """
    Parse oha output file to extract p10, p25, p50, p75, p90, p95, p99, average, fastest, slowest
    and histogram.
    Returns a dict with these metrics.
    """
    metrics = {}
    hist = []

    with open(filepath, 'r') as f:
        content = f.read()

    # Response time distribution
    dist_match = re.findall(r'\s*(\d+\.\d+)% in (\d+\.\d+) sec', content)
    for perc, val in dist_match:
        metrics[f'p{perc.split(".")[0]}'] = float(val) * 1000  # convert to ms

    # Average, fastest, slowest
    avg_match = re.search(r'Average:\s*([\d.]+) sec', content)
    if avg_match:
        metrics['average'] = float(avg_match.group(1)) * 1000
    fast_match = re.search(r'Fastest:\s*([\d.]+) sec', content)
    if fast_match:
        metrics['fastest'] = float(fast_match.group(1)) * 1000
    slow_match = re.search(r'Slowest:\s*([\d.]+) sec', content)
    if slow_match:
        metrics['slowest'] = float(slow_match.group(1)) * 1000

    # Histogram
    hist_matches = re.findall(r'\s*(\d+\.\d+) sec \[(\d+)\]', content)
    for t, count in hist_matches:
        hist.append({'time_ms': float(t)*1000, 'count': int(count)})

    metrics['histogram'] = pd.DataFrame(hist)
    return metrics




In [3]:
# -----------------------------
# Load files
# -----------------------------
result_dir = Path("D:/Projekty/studia/lsc/lsc-lab4-aws-cloud-r2-ZuzGom/results")

files_to_load = {
    "lambda_zip": result_dir / "scenario-a-zip.txt",
    "lambda_container": result_dir / "scenario-a-container.txt",
    "lambda_zip_b": result_dir / "scenario-b-lambda-zip-c10.txt",
    "lambda_container_b": result_dir / "scenario-b-lambda-container-c10.txt",
    "fargate_b": result_dir / "scenario-b-fargate-c50.txt",
    "ec2_b": result_dir / "scenario-b-ec2-c50.txt"
}

metrics_data = {k: parse_oha_file(v) for k, v in files_to_load.items()}


In [4]:
# -----------------------------
# Example 1: Latency comparison bar (Scenario A)
# -----------------------------
df_latency = pd.DataFrame([
    {
        'Target': 'Lambda Zip',
        'Cold/Warm': 'cold',  # placeholder, can extend with cloudwatch Init Duration
        'p50': metrics_data['lambda_zip']['p50'],
        'p95': metrics_data['lambda_zip']['p95'],
        'p99': metrics_data['lambda_zip']['p99'],
    },
    {
        'Target': 'Lambda Container',
        'Cold/Warm': 'cold',
        'p50': metrics_data['lambda_container']['p50'],
        'p95': metrics_data['lambda_container']['p95'],
        'p99': metrics_data['lambda_container']['p99'],
    },
])

df_latency_melt = df_latency.melt(id_vars=['Target', 'Cold/Warm'], value_vars=['p50','p95','p99'],
                                  var_name='Percentile', value_name='Latency_ms')

plt.figure(figsize=(8,5))
sns.barplot(data=df_latency_melt, x='Target', y='Latency_ms', hue='Percentile')
plt.title("Scenario A: Latency Percentiles (cold start)")
plt.ylabel("Latency (ms)")
plt.show()

# -----------------------------
# Example 2: Histogram plot (Scenario B)
# -----------------------------
plt.figure(figsize=(10,5))
for target in ['lambda_zip_b', 'lambda_container_b', 'fargate_b', 'ec2_b']:
    hist_df = metrics_data[target]['histogram']
    plt.bar(hist_df['time_ms'], hist_df['count'], width=5, alpha=0.5, label=target.replace('_b',''))

plt.xlabel("Response Time (ms)")
plt.ylabel("Request Count")
plt.title("Scenario B: Response Time Histogram")
plt.legend()
plt.show()

KeyError: 'p50'